# MSTR 用户名批量小写化

按顺序执行下面的 cell。每一步会显示当前结果，真正修改用户前会再次确认。

## 1. 导入依赖与脚本函数

如果这里报 `ModuleNotFoundError`，请先在当前环境安装依赖：`pip install -r requirements.txt`。

In [ ]:
import os
from getpass import getpass

import pandas as pd
from IPython.display import display
from mstrio.connection import Connection
from mstrio.users_and_groups.user import User, list_users

from mstr_user_lowercase import (
    apply_changes,
    build_changes,
    filter_system_users,
    find_uppercase_users,
    users_to_dataframe,
    validate_no_login_collisions,
)

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print(f"pandas version: {pd.__version__}")

## 2. 配置并登录 MSTR Library

密码不会写入 Notebook 文件。优先读取环境变量 `MSTR_PASSWORD`，没有时会在运行 cell 时提示输入。

In [ ]:
library_url = os.getenv("MSTR_LIBRARY_URL", "https://env-363414.customer.cloud.microstrategy.com/MicroStrategyLibrary/")
username = os.getenv("MSTR_USERNAME", "mstr")
password = os.getenv("MSTR_PASSWORD") or getpass("请输入 MSTR 登录密码: ")
login_mode = int(os.getenv("MSTR_LOGIN_MODE", "1"))

connection = Connection(
    base_url=library_url,
    username=username,
    password=password,
    login_mode=login_mode,
)

print(f"已连接: {library_url}")

## 3. 列出当前平台全部用户并导入 DataFrame

In [ ]:
users = list_users(connection=connection)
users_df = users_to_dataframe(users)

print(f"当前平台用户数量: {len(users_df)}")
display(users_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

## 4. 筛选包含大写字母的用户

In [ ]:
uppercase_df = find_uppercase_users(users_df)

print(f"包含大写字母的用户数量: {len(uppercase_df)}")
display(uppercase_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

## 5. 识别并确认是否剔除疑似平台自带用户

默认剔除。若想保留，把 `exclude_system_users` 改成 `False` 后重新执行本 cell。

In [ ]:
exclude_system_users = True

candidate_df, system_df = filter_system_users(uppercase_df)

print(f"疑似平台自带用户数量: {len(system_df)}")
display(system_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

target_df = candidate_df if exclude_system_users else uppercase_df

print(f"进入下一步的待处理用户数量: {len(target_df)}")
display(target_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

## 6. 生成小写化预览并检查登录 ID 冲突

数字会原样保留，只有字母会转成小写。

In [ ]:
changes = build_changes(target_df)
validate_no_login_collisions(users_df, changes)

preview_df = pd.DataFrame([change.__dict__ for change in changes])

print(f"待小写化用户数量: {len(preview_df)}")
display(preview_df[[
    "display_name",
    "login_id",
    "trust_id",
    "new_display_name",
    "new_login_id",
    "new_trust_id",
]].fillna(""))

## 7. 最终确认并执行修改

先保持 `dry_run = True` 预演。确认无误后，把 `dry_run` 改成 `False`，并把 `confirm_execute` 改成 `True`。

In [ ]:
dry_run = True
confirm_execute = False

if not dry_run and not confirm_execute:
    raise RuntimeError("真正执行前，请先把 confirm_execute 改成 True。")

result_df = apply_changes(connection, changes, dry_run=dry_run)

print("预演完成，未修改平台用户。" if dry_run else "执行完成。")
display(result_df.fillna(""))

## 8. 再次列出当前平台用户并检查结果

如果上一步是 dry-run，这里会看到目标用户仍然保持原状。

In [ ]:
refreshed_users = list_users(connection=connection)
refreshed_df = users_to_dataframe(refreshed_users)

print(f"当前平台用户数量: {len(refreshed_df)}")
display(refreshed_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

remaining_targets = find_uppercase_users(refreshed_df)
processed_ids = {change.id for change in changes}
remaining_processed = remaining_targets[remaining_targets["id"].isin(processed_ids)]

print(f"本批次目标用户中仍包含大写字母的数量: {len(remaining_processed)}")
display(remaining_processed[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

## 9. 成功确认

当上一步显示本批次目标用户中仍包含大写字母的数量为 `0`，说明批处理成功。恭喜，批处理脚本运行成功！